In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [4]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
        'POR',
        'CurrentCompletion'
    ]


In [5]:
query = f"""DROP VIEW IF EXISTS Weekly_KPI;"""
cursor.execute(query)
conn.commit()


In [6]:
query = """
CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [7]:
print(query)


CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WH

In [8]:
query = "SELECT * FROM Weekly_KPI WHERE Year = 2026 AND BoundaryRegion IS NULL;"
df = pd.read_sql_query(query, conn)
conn.close()

In [9]:
Query(query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'Cadent') AND KPIId = 'POR'").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,POR_Cadent_Y2026_W14,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,14,127000.0,None,2026-07-20 14:48:42.939082
1,POR_Cadent_Y2026_W15,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,15,127000.0,None,2026-07-20 14:48:42.939082
2,POR_Cadent_Y2026_W16,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,16,127000.0,None,2026-07-20 14:48:42.939082
3,POR_Cadent_Y2026_W17,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,17,127000.0,None,2026-07-20 14:48:42.939082
4,POR_Cadent_Y2026_W18,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,18,127000.0,None,2026-07-20 14:48:42.939082
5,POR_Cadent_Y2026_W19,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,19,127000.0,None,2026-07-20 14:48:42.939082
6,POR_Cadent_Y2026_W20,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,20,127000.0,None,2026-07-20 14:48:42.939082
7,POR_Cadent_Y2026_W21,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,21,127000.0,None,2026-07-20 14:48:42.939082
8,POR_Cadent_Y2026_W22,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,22,127000.0,None,2026-07-20 14:48:42.939082
9,POR_Cadent_Y2026_W23,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,23,127000.0,None,2026-07-20 14:48:42.939082


In [10]:
df

,Year,PeriodValue,CustomerName,BoundaryRegion,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,ServicePipeKm,...,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare,POR,CurrentCompletion
0,2026,1,Thuega Energienetze,None,86.08,84.24,56.88,55.76,NaN,29.20,...,0.04,19.05,4.76,66.67,9.52,50.00,15.62,34.38,NaN,NaN
1,2026,1,Cadent,None,50.53,47.68,49.69,47.03,NaN,0.84,...,0.06,23.08,0.00,73.63,3.30,73.47,19.39,7.14,NaN,NaN
2,2026,2,EWE,None,247.41,230.40,157.93,149.56,NaN,89.48,...,0.03,29.41,2.94,52.94,14.71,4.96,23.14,71.90,NaN,NaN
3,2026,2,NBB,None,40.72,38.39,26.84,25.75,NaN,13.88,...,0.04,36.84,5.26,52.63,5.26,37.93,27.59,34.48,NaN,NaN
4,2026,2,Thuega Energienetze,None,112.28,106.48,76.13,72.52,NaN,36.16,...,0.00,30.77,0.00,69.23,0.00,38.89,33.33,27.78,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,2026,29,Avacon,None,166.04,158.26,106.69,101.48,NaN,59.35,...,0.04,32.26,0.00,54.84,12.90,25.00,34.62,40.38,NaN,NaN
251,2026,30,EWE,None,98.49,96.87,47.52,47.10,NaN,50.97,...,0.02,0.00,0.00,92.31,7.69,7.14,39.29,53.57,NaN,NaN
252,2026,30,Cadent,None,272.56,259.36,265.31,253.14,46561.495797,7.25,...,0.11,15.97,1.52,71.48,11.03,61.24,16.57,22.19,127000.0,36.66
253,2026,30,Westnetz,None,77.62,76.55,53.70,52.86,NaN,23.91,...,0.15,0.00,0.00,55.56,44.44,5.41,43.24,51.35,NaN,NaN
